# Notebook 00 — Reconocimiento del dataset

**Objetivo de este notebook:** entender qué datos tenemos antes de tocar una sola línea de análisis.
El reconocimiento es la fase más importante de cualquier investigación forense: si no sabemos qué
hay en el dataset, cualquier análisis posterior será ciego.

**Al terminar este notebook vas a saber:**
- Qué es IronMarch y de dónde vienen estos datos
- Qué tablas hay en el dump SQL y qué contiene cada una
- Cuántos registros hay y si el número tiene sentido
- Cómo se distribuye la actividad (usuarios, posts, fechas)
- Cuáles son las preguntas de investigación que vamos a responder

**Público objetivo:** este notebook está escrito para personas con conocimientos básicos de Python.
Cada bloque de código tiene una explicación antes. No te saltes los markdowns — son parte del análisis.

## 1. ¿Qué es IronMarch?

IronMarch fue un foro de internet neonazi activo entre **2011 y 2017**. Fue fundado por Alexander
Slavros (nombre real: Alisher Mukhitdinov, de origen ruso-uzbeko) y sirvió como punto de encuentro
para militantes de extrema derecha de todo el mundo.

A diferencia de foros más populistas como 4chan o Stormfront, IronMarch tenía una orientación
filosófica explícita — fuertemente influenciada por Julius Evola, el fascismo europeo de entreguerras
y el aceleracionismo. Varios de sus miembros fueron posteriormente vinculados a atentados terroristas.

### El leak

En **noviembre de 2019**, un actor anónimo (en una operación llamada "Operation Payback" / "antifa
hackers") filtró la base de datos completa del foro. El dump SQL contiene todo: usuarios con emails,
contraseñas hasheadas, mensajes públicos y **mensajes privados** — dato extremadamente infrecuente
en leaks de foros.

El archivo que vamos a analizar es `IronMarch_2019.11.zip`, que contiene el dump en formato
SQL de **Invision Power Suite 4.x (IPS 4.x)**, el software de foros que usaba IronMarch.

### Consideración ética

Este dataset contiene información de personas reales. El análisis que hacemos aquí tiene
propósito **académico y defensivo**: entender cómo operan estas redes. No publicamos perfiles
individuales ni información de identificación personal más allá de lo que ya es público.

## 2. Imports y configuración inicial

Antes de poder hacer cualquier cosa, necesitamos cargar las herramientas que vamos a usar.

- `pandas` es la librería estándar para trabajar con tablas de datos en Python
- `numpy` nos da herramientas matemáticas (promedios, logaritmos, etc.)
- `matplotlib` y `seaborn` son para hacer gráficas
- `load_forum` es una función propia de este proyecto que sabe cómo leer el dump SQL de IronMarch
- `RESULTS_DIR` y `DATA_DIR` son rutas a los directorios del proyecto

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Agregar el directorio raíz del proyecto al path para poder importar src/
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_forum, RESULTS_DIR, DATA_DIR

# Estilo visual oscuro para todas las gráficas
plt.style.use('dark_background')
sns.set_palette('muted')

# Rutas de resultados y datos
IM_RESULTS = RESULTS_DIR / 'ironmarch'
IM_RESULTS.mkdir(parents=True, exist_ok=True)  # Crear carpeta si no existe
IM_ZIP = DATA_DIR / 'Far Right Forum' / 'IronMarch_2019.11.zip'

print(f'Datos:      {DATA_DIR}')
print(f'Resultados: {IM_RESULTS}')
print(f'ZIP existe: {IM_ZIP.exists()}')

## 3. Estructura del dump SQL

IronMarch usa **IPS 4.x** (Invision Power Suite), un software comercial de foros muy común.
Su base de datos tiene un esquema relacional que podemos representar así:

```
forum → thread → post → user
                          ↑
                   private_message
```

Lo que esto dice es:
- Un **foro** (sección del sitio) contiene muchos **hilos** (threads)
- Cada **hilo** contiene muchos **posts** (mensajes)
- Cada **post** fue escrito por un **usuario**
- Los **mensajes privados** van directamente entre usuarios

**Cardinalidades aproximadas** (lo que esperamos encontrar):
~138 foros, ~14K threads, ~194K posts, ~18K usuarios, ~55K mensajes privados.

La celda de abajo carga el dump completo. Puede tardar 1-2 minutos porque el ZIP es grande.

In [ ]:
# load_forum() detecta automáticamente el formato (IPS 4.x en este caso)
# y retorna un diccionario donde cada llave es el nombre de una tabla
# y cada valor es un DataFrame de pandas con los datos de esa tabla
raw = load_forum(IM_ZIP)

# Extraer cada tabla con un nombre legible
forums  = raw.get('forum', pd.DataFrame())
threads = raw.get('thread', pd.DataFrame())
posts   = raw.get('post', pd.DataFrame())
users   = raw.get('user', pd.DataFrame())
pms     = raw.get('private_message', pd.DataFrame())

# Normalizar userid a numérico — el parser a veces deja texto por errores de alineación
for df in [posts, threads, pms]:
    for col in ['userid', 'sender_id', 'receiver_id']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df.dropna(subset=[col], inplace=True)
            df[col] = df[col].astype(int).astype(str)

if 'userid' in users.columns:
    users['userid'] = pd.to_numeric(users['userid'], errors='coerce')
    users.dropna(subset=['userid'], inplace=True)
    users['userid'] = users['userid'].astype(int).astype(str)

# Convertir fechas (el formato IPS almacena timestamps Unix en segundos)
for df, col in [(posts, 'dateline'), (users, 'joindate'), (threads, 'dateline')]:
    if col in df.columns and len(df) > 0:
        df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')

print(f'Tablas cargadas exitosamente.')
print(f'Foros:             {len(forums):,}')
print(f'Hilos:             {len(threads):,}')
print(f'Posts:             {len(posts):,}')
print(f'Usuarios:          {len(users):,}')
print(f'Mensajes privados: {len(pms):,}')

## 4. Validación: ¿los números tienen sentido?

Antes de analizar, tenemos que verificar que los datos que cargamos son coherentes.
Esto se llama **validación de integridad** y es obligatorio en cualquier análisis forense.

Preguntas que vamos a responder:
1. ¿Los conteos coinciden con lo que sabemos del foro?
2. ¿Hay fechas imposibles (como el año 1970 — que es el "epoch 0" de Unix)?
3. ¿Hay usuarios sin nombre o posts sin texto?
4. ¿El rango temporal de 2011 a 2017 es consistente?

In [ ]:
print('=== VALIDACIÓN DE INTEGRIDAD ===\n')

# 1. Conteos esperados vs reales
esperado = {'foros': 138, 'hilos': 14000, 'posts': 194000, 'usuarios': 18000, 'pms': 55000}
real = {'foros': len(forums), 'hilos': len(threads), 'posts': len(posts), 'usuarios': len(users), 'pms': len(pms)}

print('Conteos esperados vs reales:')
for k in esperado:
    pct = real[k] / esperado[k] * 100
    status = 'OK' if 70 <= pct <= 130 else 'REVISAR'
    print(f'  {k:<12} esperado ~{esperado[k]:>7,}  real {real[k]:>7,}  ({pct:.0f}%)  [{status}]')

# 2. Rango temporal de posts
print()
if 'dateline' in posts.columns:
    valid_dates = posts['dateline'].dropna()
    epoch_zero = (posts['dateline'].dt.year < 2000).sum()
    print(f'Rango de fechas en posts:')
    print(f'  Mínima: {valid_dates.min()}')
    print(f'  Máxima: {valid_dates.max()}')
    print(f'  Fechas imposibles (año < 2000): {epoch_zero:,}  [{"OK" if epoch_zero == 0 else "REQUIERE LIMPIEZA"}')

# 3. Nulos en columnas críticas
print()
print('Nulos en columnas críticas:')
text_col = 'pagetext' if 'pagetext' in posts.columns else 'message'
null_posts_text = posts[text_col].isna().sum() if text_col in posts.columns else 'N/A'
null_userid_posts = posts['userid'].isna().sum() if 'userid' in posts.columns else 'N/A'
null_username = users['username'].isna().sum() if 'username' in users.columns else 'N/A'
print(f'  Posts sin texto:    {null_posts_text}')
print(f'  Posts sin userid:   {null_userid_posts}')
print(f'  Usuarios sin nombre: {null_username}')

## 5. Distribuciones básicas: primer vistazo

### 5.1 ¿Quiénes escriben más?

En casi todos los foros de internet existe un patrón llamado **power-law** (ley de potencia):
una minoría pequeña de usuarios genera la mayoría del contenido, y la gran mayoría apenas escribe.

La gráfica de la izquierda muestra la distribución normal. La de la derecha usa escala
logarítmica en ambos ejes — si la distribución forma una línea recta en log-log, confirmamos
que es una power-law.

In [ ]:
if 'userid' in posts.columns:
    # Contar posts por usuario
    post_counts = posts.groupby('userid').size().sort_values(ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Gráfica izquierda: distribución en escala normal
    axes[0].hist(post_counts.values, bins=60, color='#E94E4E', alpha=0.85)
    axes[0].set_title('Distribución de posts por usuario')
    axes[0].set_xlabel('Cantidad de posts')
    axes[0].set_ylabel('Número de usuarios')

    # Gráfica derecha: escala log-log para ver la power-law
    bins = np.logspace(0, np.log10(post_counts.max() + 1), 40)
    axes[1].hist(post_counts.values, bins=bins, color='#E94E4E', alpha=0.85)
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_title('Power-law (escala log-log)')
    axes[1].set_xlabel('Posts (escala logarítmica)')
    axes[1].set_ylabel('Usuarios (escala logarítmica)')

    plt.suptitle('IronMarch — actividad de usuarios', y=1.01)
    plt.tight_layout()
    plt.show()

    # Estadísticas clave
    print(f'Total de usuarios únicos: {len(post_counts):,}')
    print(f'Usuarios con exactamente 1 post:  {(post_counts == 1).sum():,}  ({(post_counts == 1).mean()*100:.1f}%)')
    print(f'Usuarios con 10 posts o más:      {(post_counts >= 10).sum():,}  ({(post_counts >= 10).mean()*100:.1f}%)')
    print(f'Usuarios con 100 posts o más:     {(post_counts >= 100).sum():,}')
    top10_pct = post_counts.nlargest(int(len(post_counts) * 0.1)).sum() / post_counts.sum() * 100
    print(f'El 10% más activo genera el {top10_pct:.0f}% de todos los posts')

### 5.2 Los 20 usuarios más activos

Identificar los actores más prolíficos es el primer paso del análisis de actores.
En foros de extrema derecha, los usuarios muy activos suelen cumplir roles específicos:
líderes ideológicos, reclutadores, o moderadores.

In [ ]:
if 'userid' in posts.columns and 'username' in users.columns:
    uid_to_name = dict(zip(users['userid'], users['username']))

    top_users = (
        posts.groupby('userid').size()
        .sort_values(ascending=False)
        .head(20)
        .reset_index()
    )
    top_users.columns = ['userid', 'posts']
    top_users['username'] = top_users['userid'].map(uid_to_name)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(
        top_users['username'].fillna(top_users['userid'].astype(str))[::-1],
        top_users['posts'][::-1],
        color='#E94E4E', alpha=0.85,
    )
    ax.set_xlabel('Posts totales')
    ax.set_title('Top 20 usuarios más activos — IronMarch')
    plt.tight_layout()
    plt.show()

    print('Top 10 usuarios más activos:')
    print(top_users[['username', 'posts']].head(10).to_string(index=False))

### 5.3 Evolución temporal: ¿cuándo era más activo el foro?

Ver la actividad a lo largo del tiempo nos permite detectar dos cosas:
1. **El ciclo de vida del foro**: ¿cuándo creció, cuándo declinó?
2. **Picos de actividad**: ¿correlacionan con eventos externos (atentados, elecciones)?

Marcamos con líneas verticales algunos eventos clave para orientarnos.

In [ ]:
EVENTS = {
    '2011-09': 'Fundación del foro',
    '2017-11': 'Cierre',
}

if 'dateline' in posts.columns:
    valid_posts = posts.dropna(subset=['dateline'])
    valid_posts = valid_posts[
        (valid_posts['dateline'].dt.year >= 2011) &
        (valid_posts['dateline'].dt.year <= 2018)
    ]
    # Agrupar por mes — dt.to_period('M') convierte cada fecha al mes correspondiente
    monthly = valid_posts.groupby(valid_posts['dateline'].dt.to_period('M')).size()

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(range(len(monthly)), monthly.values, color='#E94E4E', linewidth=1.5)
    ax.fill_between(range(len(monthly)), monthly.values, alpha=0.2, color='#E94E4E')

    month_index = {str(p): i for i, p in enumerate(monthly.index)}
    for event_ym, label in EVENTS.items():
        if event_ym in month_index:
            idx = month_index[event_ym]
            ax.axvline(idx, color='#FFD700', alpha=0.7, linestyle='--', linewidth=1.5)
            ax.text(idx + 0.3, monthly.max() * 0.9, label,
                    rotation=90, fontsize=8, color='#FFD700', va='top')

    tick_step = max(1, len(monthly) // 14)
    ax.set_xticks(range(0, len(monthly), tick_step))
    ax.set_xticklabels(
        [str(monthly.index[i]) for i in range(0, len(monthly), tick_step)],
        rotation=45, fontsize=8,
    )
    ax.set_title('IronMarch — posts por mes (2011–2018)')
    ax.set_ylabel('Posts por mes')
    plt.tight_layout()
    plt.show()

    print(f'Mes más activo: {monthly.idxmax()} con {monthly.max():,} posts')
    print(f'Mes menos activo (excl. primeros 6 meses): {monthly[6:].idxmin()} con {monthly[6:].min():,} posts')

### 5.4 Distribución por hora y día de la semana

Este heatmap muestra en qué horas UTC y días de la semana se publicaban más posts.
¿Por qué importa? Porque la franja horaria dominante nos da información sobre la
**zona geográfica** de los usuarios. Si la mayoría postea entre las 14h y 22h UTC,
es probable que estén en América del Norte (9am–5pm hora del este).

El eje X es la hora UTC (0–23) y el eje Y es el día de la semana.

In [ ]:
if 'dateline' in posts.columns:
    df_tz = posts.dropna(subset=['dateline']).copy()
    df_tz['hour']    = df_tz['dateline'].dt.hour
    df_tz['weekday'] = df_tz['dateline'].dt.weekday  # 0=Lunes, 6=Domingo

    # Contar posts por combinación (día, hora)
    heatmap_data = df_tz.groupby(['weekday', 'hour']).size().unstack(fill_value=0)
    day_names = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
    heatmap_data.index = [day_names[i] for i in heatmap_data.index]

    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(heatmap_data, cmap='YlOrRd', ax=ax, linewidths=0.1,
                cbar_kws={'label': 'Posts'})
    ax.set_title('IronMarch — actividad por hora UTC y día de la semana')
    ax.set_xlabel('Hora UTC')
    plt.tight_layout()
    plt.show()

    # ¿Qué hora tiene el pico global?
    peak_hour = df_tz.groupby('hour').size().idxmax()
    peak_day  = df_tz.groupby('weekday').size().idxmax()
    print(f'Hora UTC de mayor actividad: {peak_hour:02d}h')
    print(f'Día de mayor actividad: {day_names[peak_day]}')
    print(f'Interpretación: hora {peak_hour}h UTC = {peak_hour - 5}h EST / {peak_hour - 8}h PST')

### 5.5 Secciones del foro más activas

Los foros IPS no guardan el `forumid` directamente en cada post. En cambio, cada post tiene
un `threadid`, y cada hilo tiene un `forumid`. Entonces tenemos que hacer un **join**:
cruzar la tabla de posts con la tabla de hilos para saber a qué sección pertenece cada post.

Este join funciona como buscar en un índice: dado un `threadid`, busca en la tabla de hilos
cuál es su `forumid`, y desde ahí ya podemos buscar el nombre del foro.

In [ ]:
if len(threads) > 0 and 'forumid' in threads.columns and 'threadid' in posts.columns:
    # Crear diccionario threadid → forumid para el join
    thread_to_forum = threads.set_index('threadid')['forumid'].to_dict()
    posts_with_forum = posts.copy()
    posts_with_forum['forumid'] = posts_with_forum['threadid'].map(thread_to_forum)

    if len(forums) > 0:
        fname_col = next((c for c in forums.columns if c == 'name'), None)
        fid_col   = next((c for c in forums.columns if c == 'forumid'), None)

        if fname_col and fid_col:
            fid_to_name = dict(zip(forums[fid_col].astype(str), forums[fname_col]))
            posts_with_forum['forumid'] = posts_with_forum['forumid'].astype(str)

            forum_counts = (
                posts_with_forum.groupby('forumid').size()
                .sort_values(ascending=False)
                .head(20)
                .reset_index()
            )
            forum_counts.columns = ['forumid', 'posts']
            forum_counts['name'] = forum_counts['forumid'].map(fid_to_name).fillna('(desconocido)')

            fig, ax = plt.subplots(figsize=(10, 8))
            ax.barh(forum_counts['name'][::-1], forum_counts['posts'][::-1],
                    color='#E94E4E', alpha=0.85)
            ax.set_xlabel('Posts')
            ax.set_title('Top 20 secciones del foro más activas — IronMarch')
            plt.tight_layout()
            plt.show()

            print('Top 5 secciones:')
            print(forum_counts[['name', 'posts']].head(5).to_string(index=False))
else:
    print('No se pudo completar el join posts ↔ foros.')
    print(f'Columnas en posts: {list(posts.columns)}')
    print(f'Columnas en threads: {list(threads.columns) if len(threads) > 0 else "vacío"}')

## 6. Distribución de usuarios por fecha de registro

La curva de registros nos muestra cuándo llegaron nuevos miembros al foro.
Un pico de registros puede indicar un evento de reclutamiento o una mención mediática.
Un declive sostenido indica que el foro perdía atractivo.

In [ ]:
if 'joindate' in users.columns:
    valid_joins = users.dropna(subset=['joindate'])
    valid_joins = valid_joins[
        (valid_joins['joindate'].dt.year >= 2011) &
        (valid_joins['joindate'].dt.year <= 2018)
    ]
    # Agrupar por trimestre para suavizar el ruido
    quarterly = valid_joins.groupby(valid_joins['joindate'].dt.to_period('Q')).size()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(range(len(quarterly)), quarterly.values, color='#4E9EE9', alpha=0.85)
    tick_step = max(1, len(quarterly) // 10)
    ax.set_xticks(range(0, len(quarterly), tick_step))
    ax.set_xticklabels(
        [str(quarterly.index[i]) for i in range(0, len(quarterly), tick_step)],
        rotation=45, fontsize=8,
    )
    ax.set_title('Nuevos usuarios registrados por trimestre — IronMarch')
    ax.set_ylabel('Nuevos registros')
    plt.tight_layout()
    plt.show()

    print(f'Total usuarios con fecha de registro válida: {len(valid_joins):,}')
    print(f'Trimestre con más registros: {quarterly.idxmax()} ({quarterly.max()} usuarios)')
else:
    print('Sin columna joindate en usuarios.')

## 7. Preguntas de investigación

Con el reconocimiento del dataset completo, podemos formular preguntas de investigación
específicas y verificables. Estas preguntas van a guiar los notebooks siguientes.

---

### P1 — ¿Quiénes son los actores clave?
**Hipótesis:** hay una minoría de usuarios (probablemente < 5%) que genera la mayoría del
contenido y ocupa posiciones centrales en la red. Algunos tienen roles diferenciados:
líderes ideológicos, reclutadores, moderadores, usuarios nuevos en proceso de radicalización.

**Técnicas:** betweenness centrality (red pública + PMs), NER por usuario, topic modeling.
→ *Notebooks 02 y 03*

---

### P2 — ¿Hay cuentas dobles (sockpuppets)?
**Hipótesis:** algunos usuarios operan múltiples cuentas — ya sea para evadir bans,
amplificar su influencia, o mantener una presencia anónima separada de su cuenta pública.

**Técnicas:** Burrows' Delta (estilometría), similitud de patrones temporales, análisis de PMs.
→ *Notebook 03*

---

### P3 — ¿Qué eventos externos correlacionan con la actividad del foro?
**Hipótesis:** atentados terroristas, elecciones importantes o eventos de extrema derecha
generaron picos de actividad y cambios en el vocabulario dominante.

**Técnicas:** serie temporal semanal + z-score, análisis de frecuencias de palabras clave por período.
→ *Notebook 02*

---

### P4 — ¿Hay estructura de comunidades dentro del foro?
**Hipótesis:** el foro no es una comunidad homogénea — hay subcomunidades con vocabularios
y referentes ideológicos distintos (por ejemplo: neonazis clásicos vs. aceleracionistas).

**Técnicas:** análisis de comunidades Louvain en la red, clusters UMAP+HDBSCAN en espacio de embeddings.
→ *Notebooks 02 y 03*

---

Estas cuatro preguntas van a quedar respondidas (con sus limitaciones) en el **Notebook 04 — Síntesis**.

In [ ]:
# Resumen final del reconocimiento — para tener siempre a mano los números clave
print('='*55)
print('RESUMEN DE RECONOCIMIENTO — IronMarch')
print('='*55)
print(f'Foros (secciones):    {len(forums):,}')
print(f'Hilos:                {len(threads):,}')
print(f'Posts:                {len(posts):,}')
print(f'Usuarios:             {len(users):,}')
print(f'Mensajes privados:    {len(pms):,}')

if 'dateline' in posts.columns:
    valid = posts['dateline'].dropna()
    if len(valid):
        print(f'Rango temporal:       {valid.min().strftime("%Y-%m-%d")} → {valid.max().strftime("%Y-%m-%d")}')

print()
print('Preguntas de investigación definidas: 4')
print('  P1 — Actores clave (notebook 03)')
print('  P2 — Sockpuppets (notebook 03)')
print('  P3 — Correlación con eventos externos (notebook 02)')
print('  P4 — Estructura de comunidades (notebooks 02 y 03)')
print()
print('Siguiente paso: 01_ingenieria_datos.ipynb')